[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eirasf/GCED-AA3/blob/main/en/lab4/lab4.ipynb)

# Lab4: Reinforcement learning - Dynamic programming

In this lab we will get familiar with [Gym](https://www.gymlibrary.dev/), a Python library to simulate problems that can be solved using reinforcement learning.
In addition, we will develop some control methods based on dynamic programming.

## Gym
Gym is a tool to develop and compare reinforcement learning algorithms. It facilitates the simulation of interactions of an agent with very diverse environments.

To use OpenGym, the first step is to import the library. Once done, we will be able to create an environment that allows us to simulate the problem we want from among the many available.

In this practical we will use a very simple environment that simulates a *GridWorld*, that is, a world made of cells through which the agent will be able to move. In particular, we will load the environment `MiniGrid-Empty-5x5-v0`.

In [ ]:
# If the packages are not installed, you have to run these lines:
#!pip install gymnasium[classic-control]
#!pip install minigrid
import gymnasium as gym
import minigrid
import numpy as np
env = gym.make('MiniGrid-Empty-5x5-v0', render_mode='rgb_array')

The `gym.make` method returns an environment type object that offers us, among others, the following methods and properties:
 - `reset`: Returns the environment to its original state
 - `actions`: Shows a list of the available actions
 - `max_steps`: Sets the maximum number of actions that the agent can perform in each episode
 - `render`: Returns an image where the current situation is represented
 - `step(action)`: Executes an action and updates the environment accordingly

The goal of the agent in this problem is to reach the goal, represented by a green cell. The agent is represented as a red arrow that reflects its position and orientation.

In [ ]:
# Show the available actions
actions = env.get_wrapper_attr('actions')
print([a.name for a in actions])

# Reset the environment
env.reset()

# Show the environment in its initial state
import matplotlib.pyplot as plt

def show_environment(env:gym.Env) -> None:
    im = plt.imshow(env.render())
    plt.show()
show_environment(env)

# TODO: Execute the forward action (you can refer to the i-th action as actions(i) or as actions.name) and show the environment again
...

# TODO: Try to see how the different actions affect the environment
...

The [`step(action)`](https://gymnasium.farama.org/api/env/#gymnasium.Env.step) method returns five values:
 - Observation: Represents what the agent can perceive from the environment in its current state
 - Reward: Indicates the value of the reward signal for the execution of the action applied in the state in which it was applied.
 - Terminated: Boolean value that indicates whether the episode has finished
 - Truncated: Boolean value that indicates whether the episode has reached the maximum number of steps
 - Info: Additional information

In order to learn a policy, we must find a way to represent the states. Usually, the agent would base its state representation on its observations. However, for this particular problem, we are going to use a simpler state representation than the observation that the agent can make. This state representation task would be carried out by the **interpreter** in the usual scheme of a reinforcement learning problem.
![Reinforcement learning scheme](img/Reinforcement_learning_diagram.svg)

The interpreter will identify the state with a three-component vector that will indicate, respectively, the column, row and orientation of the agent. The rows/columns will be numbered from 0 to 2 (from top to bottom and left to right) and the orientation can take the following values:
 - 0 $\rightarrow$ right
 - 1 $\rightarrow$ down
 - 2 $\rightarrow$ left
 - 3 $\rightarrow$ up

The environment of this problem gives us direct access to two variables that will be useful:
 - `agent_pos`: Indicates the cell where the agent is. The cells of the board are numbered from 1 to 3, so it will be necessary to adapt that numbering.
 - `agent_dir`: Indicates the orientation of the agent using the code described in the previous paragraph.

 You can access these values as `env.get_wrapper_attr('agent_pos')` and `env.get_wrapper_attr('agent_dir')` respectively.
Let's create a function that carries out the interpreter tasks and returns the encoding of the current state of the environment:

In [ ]:
from collections import namedtuple

# We will represent the states as tuples with the following named fields:
#  - x: Column where the agent is located. The leftmost visitable column will be 0
#  - y: Row where the agent is located. The topmost visitable row will be 0
#  - dir: Direction the agent is facing. We will respect the numeric code indicated by agent_dir
State = namedtuple('State', ['x', 'y', 'dir'])

def get_state(env:gym.Env) -> State:
    # TODO - Complete the function
    ...

env.reset()

# CHECK
current_state = get_state(env)
assert current_state.x == 0, f'The state right after resetting must indicate x=0 and yours indicates {current_state.x}'
assert current_state.y == 0, f'The state right after resetting must indicate y=0 and yours indicates {current_state.y}'
assert current_state.dir == 0, f'The state right after resetting must indicate dir=0 and yours indicates {current_state.dir}'

### Simulating an episode with a random policy

Now that we know how to handle the environment, we are going to test how well a random policy works.

Create a loop that simulates a complete episode following a random policy, that is, that applies a random action until the environment tells us that the episode has ended. You can obtain a random action by calling `env.action_space.sample()`.

Show the environment after applying each action. Also show a message that indicates:
 - Which step number has been executed
 - The name of the applied action
 - The resulting state
 - The reward obtained in that step
 - If the execution has finished

In [ ]:
# We remove the step limit per episode. The episode will only end when reaching the goal.
env.max_steps = float('inf')
print(f'Max steps modified: {env.max_steps}')


completed = False
counter = 0
while not completed:
    # TODO - Complete the loop
    ...

### Simplification of the action space and definition of policies
You will have noticed that there are actions that are not useful for this problem. We are going to simplify the search by reducing the action space to three:
 1. left
 2. right
 3. forward

By doing this, we can no longer generate a random action using `env.action_space.sample()`. Furthermore, that function chooses a random action, which does not interest us. Our goal is to obtain a **policy** that determines the best action in each situation, so we are going to take the opportunity to define a data structure that allows us to do that.

The policy must indicate a probability $\pi(a|s)$ of choosing action $a$ being in state $s$ (so $\sum_{a'}\pi(a'|s)=1$, with $s\in \mathcal{S}$ and $a\in \mathcal{A}$). We will declare a policy as a `numpy.array` that associates to each state $s$ a probability distribution over the different actions.

The random policy will give the same probability to all the actions for each state (as there are three actions, for any state-action pair the probability will be 1/3).

In [ ]:
# We select only the three indicated actions
USEFUL_ACTIONS = [actions.left, actions.right, actions.forward]


# TODO - Indicate the shape of the policy
random_policy = np.zeros(...)
# We initialize all the values to 1/3
random_policy[:] = 1.0/3

Once we have defined the policy, we can define a function to replace `env.action_space.sample()`. This function will receive the policy to follow and the state for which to select the action. It will return the action sampled following the distribution described by the policy for that state.

In [ ]:
def sample_action(state:State, policy:np.ndarray) -> int:
    # We obtain the probability distribution
    probs = policy[state.x, state.y, state.dir, :]
    # Check that the policy is correct (it can be commented out when we are sure it is)
    np.testing.assert_almost_equal(np.sum(probs), 1.0, err_msg='The policy for a state must always sum to 1')
    # Return a random action sampled according to the probabilities indicated by the policy
    return np.random.choice(USEFUL_ACTIONS, p=probs)

# TODO - Obtain the action sampled from the random_policy for the state in which the agent is in the central cell facing down
action = ...

print(action)

### Simulation of episodes following a predefined policy
Now that we have a definition for policies and a sampling function over any policy, we can adapt our loop that simulates an episode so that it does it following a specific policy.

We will define a function called `simulate_episode` for that. The function must receive the policy to follow and two parameters that will indicate, respectively, whether the *renders* of the episode steps will be shown and whether text messages will be shown.

The function must return a tuple with the **return** obtained, that is, the sum of the rewards received in each step, and the **number of steps** needed to complete the episode.

In [ ]:
def simulate_episode(policy:np.ndarray, show_renders:bool=False, verbose:bool=False) -> tuple[float, int]:
    env.reset()
    # TODO - Adapt the loop you wrote 3 code cells above so that it uses the policy to decide which action to take in each step
    ...
    return (G, counter)

G, num_steps = simulate_episode(random_policy, show_renders=False, verbose=True)
print(f'Simulated an episode with return {G} ({num_steps})')

### Checking the performance of the policy

Let's do an experiment to evaluate the effectiveness of the random policy. We will simulate 200 episodes, recording the number of steps needed to complete each one. It is not necessary to keep track of the returns obtained because in this problem it will always be 1 (since only the last action gives a reward, with value 1).

We will show statistics and plots that can help us get an idea of how well it works.

In [ ]:
def check_policy(policy:np.ndarray) -> None:
    episode_steps = []

    # TODO - Simulate 200 episodes, adding their respective numbers of steps to episode_steps and showing for each one a message of the form '{iteration} - Simulated an episode with return {G} ({num_steps})'
    ...

    assert(len(episode_steps)==200)

    # We show a histogram for the duration of the episodes
    plt.hist(episode_steps)
    plt.show()

    # We also show a box plot
    plt.boxplot(episode_steps)
    plt.show()

    # Finally, we report a couple of statistics
    print('On average,',np.mean(episode_steps),'steps were needed (+-',np.std(episode_steps),')')
    print('The shortest episode lasted',np.min(episode_steps),'steps and the longest episode lasted',np.max(episode_steps),'steps')

check_policy(random_policy)

# Obtaining optimal policies
## Dynamic programming methods

When it comes to finding an optimal policy to guide the actions of our agent, methods based on dynamic programming are a good option when we have knowledge about the dynamics of the environment.

In particular, these methods require knowing $p(s',r | s,a)$, that is, the probability of ending up in a state $s'$ receiving a reward $r$ if we apply action $a$ to state $s$.

OpenGym does not provide us with this information, but in this simple problem we can reconstruct it. This environment is deterministic, which means that if we apply action $a$ to state $s$ we will always obtain the same state $s'$ and the same reward $r$ with probability 1 (any other state $s_j$ or reward $r_j$ will have $p(s_j,r_j|s,a)=0$).

We can, therefore, represent the model with two variables:
 - `transition_model`: For each combination $s,a$, stores the state $s'$ to which applying action $a$ to $s$ leads.
 - `reward_model`: For each combination $s,a$, stores the value of the reward obtained when applying action $a$ to $s$.

Knowing how the environment behaves, we define these two variables with the appropriate values.

In [ ]:
# MODEL DEFINITION

# Transition model - Stores which state each action leads to from each state
# We store it in a numpy array in which for each state we store, for each action, the state it leads to.
transition_model = np.zeros((3,3,4,3,3),dtype=np.int16)
# We go through every state to complete the state that each one of the three possible actions leads to
for i in range(3): # Columns (x)
    for j in range(3): # Rows (y)
        for k in range(4): # Orientations (dir)
            # Applying the action left rotates the agent to the left, so the resulting state will have the same value for rows and columns but the orientation will vary
            transition_model[i,j,k,0]=[i,j,(k+4-1)%4]
            # Applying the action right rotates the agent to the right, so the resulting state will have the same value for rows and columns but the orientation will vary
            transition_model[i,j,k,1]=[i,j,(k+1)%4]
        # Applying the action forward keeps the orientation, but changes rows or columns depending on where the agent is facing
        if i<2:
            transition_model[i,j,0,2]=[i+1,j,0] # Right
        if j<2:
            transition_model[i,j,1,2]=[i,j+1,1] # Down
        if i>0:
            transition_model[i,j,2,2]=[i-1,j,2] # Left
        if j>0:
            transition_model[i,j,3,2]=[i,j-1,3] # Up

# Reward model - Stores which reward applying each action to each state provides
# We store it in a numpy array in which for each state we store, for each action, the reward.
reward_model = np.zeros((3,3,4,3))
# It will be 0 always except in two cases:
# 1 - It is in the second column of the third row, facing right and the forward action is used
reward_model[1,2,0,2] = 1
# 2 - It is in the third column of the second row, facing down and the forward action is used
reward_model[2,1,1,2] = 1

# Auxiliary function to consult the states that each action leads to for a given state
def get_resulting_states_for_action(state: State) -> list[State]:
    possible_next_states = transition_model[state.x, state.y, state.dir]
    return list(map(lambda x: State(*x), possible_next_states))

# Auxiliary function to consult the rewards of applying each action in a given state
def get_rewards_for_action(state: State) -> np.ndarray:
    # TODO - Complete the function
    return reward_model[state.x, state.y, state.dir]

# TODO - Show the resulting states and the rewards of applying the different actions when the agent is in the third cell of the second row, facing down and the forward action is executed
...
...

## Algorithm 1: Iterative policy evaluation
The iterative policy evaluation algorithm calculates the value $v_\pi(s)$ for every state $s$ following the policy $\pi$. The formula it uses, which is based on the Bellman equation for $v_\pi(s)$, is this:

$$v_\pi(s)=\mathbb{E}_\pi [R_{t+1}+\gamma v_k(S_{t+1}) | S_t=s]$$

$$=\sum_a\pi(a|s)\sum_{s',r}p(s',r|s,a)[r+\gamma v_k(s')]$$

As this is a deterministic problem, the expression is simplified, since $p(s',r|s,a)$ will be 0 in all cases except in 1 (because applying action $a$ to state $s$ always leads to one fixed and unique state $s'$ and always grants one fixed and unique reward $r$; we can obtain both values from the model).

You can see the complete algorithm here (and an example in slide 19 of Topic 3 of the theory):
![Iterative policy evaluation](./img/iterative-policy-evaluation.png)

Let's implement the algorithm to obtain the values assigned to each state following a fixed policy.

**TIP: To ensure convergence, update all the values of $V_{t+1}(s)$ from the values $V_t(s)$, that is, create a variable `new_state_values` where you store all those calculated in one iteration and when the iteration is completed set `state_values = new_state_values`**

In [ ]:
GAMMA = 0.1

def iterative_policy_evaluation(policy: np.ndarray) ->np.ndarray:
    THETA = 0.00001

    # In this variable we will store the values calculated for the states
    # TODO - Indicate the appropriate shape. You must store a value for each possible state
    state_values = np.zeros(...)

    # TODO - Complete the algorithm as described above
    ...
    return state_values

state_values = iterative_policy_evaluation(random_policy)
print(state_values)

# CHECK
np.testing.assert_almost_equal(state_values[0,0,1], 4.11522634e-07)

### Policy improvement

In the previous section we have obtained a value for each state of the problem when policy $\pi$ is followed. According to what was seen in theory, we can use these values to find a new policy $\pi'$ that is equal to or better than $\pi$.

When we have $v_\pi(s)$ and the model that describes the operation of the environment, we can obtain a policy $\pi'\geq\pi$ simply by selecting, for each state, the action that gives us the greatest return, which we can calculate as the immediate reward plus the value of the state we reach (discounted by $\gamma$).

$$\pi'(s)=\arg\max_a\sum_{s',r}p(s',r|s,a)\left[r+\gamma v_\pi(s')\right]$$

Let's write an algorithm that obtains $\pi'$ from `state_values`.

In [ ]:
def create_greedy_policy(state_values: np.ndarray) -> np.ndarray:
    new_shape = list(state_values.shape)
    new_shape.append(3)
    # TODO - Indicate the appropriate shape. For each state-action pair we must store the probability that this policy gives to taking that action in that state
    # Since the policy is deterministic, for a given state all the actions will have probability 0 except one that will have probability 1
    policy = np.zeros(...)
    # TODO - Use nested loops to go through all the states
    ...
                # Now for this state we must calculate, for all the actions a, G=(r + GAMMA*state_values[s[0],s[1],s[2]]) where r is the reward of applying a and s is the resulting state.
                # Of all those values, we will take the best one (let's assume it is the i-th) and put probability 1 on the i-th action for that state.
                # TODO - Go through the actions, calculating their G. Select the best one and set to 1 the probability of the corresponding action.
                ...
    return policy

improved_policy = create_greedy_policy(state_values)

# CHECK
assert(np.array_equal(improved_policy[2,1,1],[0., 0., 1.]))

## Obtaining the optimal policy - Policy iteration
Now that we can evaluate a policy and, from those values, obtain an improved policy, we can repeat the process to keep improving our policy until it does not change. This procedure is called *policy iteration* and you have it described here:

![Policy iteration](./img/policy-iteration.png)

Let's implement it (we already have most of it done).

In [ ]:
def policy_iteration(initial_policy: np.ndarray) -> np.ndarray:
    current_policy = initial_policy
    policy_stable = False
    # We will keep track of the step we are in
    step = 0
    while not policy_stable:
        # TODO - Calculate the values for the current policy
        state_values = ...
        # TODO - Obtain the new improved policy from the calculated values
        new_policy = ...
        # We check if the policy has changed
        policy_stable = np.array_equal(current_policy, new_policy)
        # We prepare the next iteration
        current_policy = new_policy
        step+=1
        print('Step #',step)
    # When the loop ends we will have obtained a policy that no longer improves
    return current_policy

optimal_policy = policy_iteration(random_policy)
print(optimal_policy)

# CHECK
assert(np.array_equal(optimal_policy[2,0,1], [0., 0., 1.]))

Let's check that the obtained policy works well by simulating an episode following it.

In [ ]:
simulate_episode(optimal_policy, show_renders=True, verbose=True)

## Obtaining the optimal policy - Value iteration
The policy iteration process is expensive because it requires making an iterative estimation of $v_\pi(s)$ for all the policies it goes through. This process can be simplified and an optimal policy can be obtained by following the value iteration algorithm:

![Value iteration](./img/value-iteration.png)

The algorithm is very similar to the iterative policy evaluation one. The difference is that, instead of calculating $\mathbb{E}_\pi [R_{t+1}+\gamma v_k(S_{t+1}) | S_t=s]$, the value of the best action is taken. At the end, the greedy policy derived from the calculated values is returned. Let's implement it.

In [ ]:
def value_iteration() -> np.ndarray: # It receives no parameters. It calculates values and policy from the environment model.
    # TODO - Repeat the iterative policy evaluation algorithm but taking the G of the best action for each state
    # Once you have the state_values, return the policy derived from them
    ...

optimal_via_values = value_iteration()
print(optimal_via_values)

# CHECK
assert(np.array_equal(optimal_via_values[2,0,1], [0., 0., 1.]))

To finish, let's check that the policy obtained this way also works well.

In [ ]:
simulate_episode(optimal_via_values, show_renders=True, verbose=True)
env.close() # We also close the environment

# Congratulations!
You have finished this lab. Now you know how to interact with OpenGym and how the main dynamic programming algorithms for reinforcement learning work.